In [7]:
# A simple, temporary version of the class for debugging
class SWOTAnalyzer:
    def __init__(self):
        pass

    def run_analysis(self):
        print("Success! The run_analysis method exists.")

# Create an instance and list its methods
test_analyzer = SWOTAnalyzer()

# Print all methods and attributes of the object
print("Methods and attributes of the test object:")
print(dir(test_analyzer))

# Now, try to run the method
test_analyzer.run_analysis()

Methods and attributes of the test object:
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'run_analysis']
Success! The run_analysis method exists.


In [4]:
import sys
import os
import xarray as xr
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pyproj import Transformer
from scipy import interpolate
import matplotlib.pyplot as plt
import earthaccess

In [5]:


# Use these lines if your util folder is in the src directory
project_src_path = "/home/jovyan/Desktop/swot-surf/src"
if project_src_path not in sys.path:
    sys.path.append(project_src_path)
import util.plotting_helpers as plothelp

auth = earthaccess.login(strategy="netrc")

class SWOTAnalyzer:
    """
    A class to perform analysis and visualization of SWOT L2 Water Mask data.
    """
    def __init__(self):
        """Initializes the analyzer."""
        self.swot_datasets = []

    def _get_granule_data(self, granule_type):
        """
        Prompts the user for SWOT granule details and returns the opened xarray Dataset.
        """
        print(f"\n--- Enter {granule_type} SWOT granule details ---")
        try:
            results = earthaccess.search_data(
                short_name="SWOT_L2_HR_Raster_C",
                granule_name=input(f"Granule name for {granule_type} (e.g., SWOT_L2_HR_Raster_...): "),
                temporal=(
                    input(f"Start date for {granule_type} (YYYY-MM-DD): "),
                    input(f"End date for {granule_type} (YYYY-MM-DD, same as start for single day): ")
                ),
            )
            print(f"Number of granules found: {len(results)}")
            if len(results) == 0:
                print("No granules found! Check your search parameters.")
                return None
            else:
                print("Granules found. Proceeding to open...")
                return xr.open_dataset(earthaccess.open(results)[0], engine="h5netcdf")
        except Exception as e:
            print(f"An error occurred during data retrieval: {e}")
            return None

    def _plot_swot_data(self, ax, ds, title, vmin, vmax):
        """
        Plots SWOT Water Surface Elevation data on a given axes.
        Returns the pcolormesh object and the WSE data array.
        """
        if ds is None:
            ax.set_title(f"{title}\n(No data available)", fontsize=14)
            ax.set_visible(False)
            return None, None

        wse = ds["wse"] + ds["height_cor_xover"]
        
        utm_zone = ds.utm_zone_num
        utm_crs = ccrs.UTM(zone=utm_zone, southern_hemisphere=False)
        transformer = Transformer.from_crs(utm_crs, ccrs.PlateCarree(), always_xy=True)
        x_utm, y_utm = wse["x"].values, wse["y"].values
        X_utm, Y_utm = np.meshgrid(x_utm, y_utm)
        X_lon, Y_lat = transformer.transform(X_utm, Y_utm)
        
        mesh = ax.pcolormesh(
            X_lon,
            Y_lat,
            wse.values,
            transform=ccrs.PlateCarree(),
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
        )
        
        ax.gridlines(draw_labels=True)
        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
        ax.coastlines(resolution='10m', linewidth=1)
        ax.set_title(title, fontsize=14)
        
        return mesh, wse

    def _plot_difference(self, ax, ds_before, ds_after, title, vmin, vmax):
        """
        Calculates and plots the difference between two SWOT WSE datasets.
        Returns the pcolormesh object.
        """
        if ds_before is None or ds_after is None:
            ax.set_title(f"{title}\n(Cannot compute difference due to missing data)", fontsize=14)
            ax.set_visible(False)
            return None

        wse_before = ds_before["wse"] + ds_before["height_cor_xover"]
        wse_after = ds_after["wse"] + ds_after["height_cor_xover"]
        
        if ds_before.utm_zone_num != ds_after.utm_zone_num:
            print(f"Warning: UTM zones differ. Reprojecting 'before' to 'after's CRS for difference calculation.")
            transformer_before_to_latlon = Transformer.from_crs(
                ccrs.UTM(zone=ds_before.utm_zone_num, southern_hemisphere=False), ccrs.PlateCarree(), always_xy=True)
            transformer_after_to_latlon = Transformer.from_crs(
                ccrs.UTM(zone=ds_after.utm_zone_num, southern_hemisphere=False), ccrs.PlateCarree(), always_xy=True)
            x_before_utm, y_before_utm = wse_before["x"].values, wse_before["y"].values
            X_before_utm, Y_before_utm = np.meshgrid(x_before_utm, y_before_utm)
            lon_before, lat_before = transformer_before_to_latlon.transform(X_before_utm, Y_before_utm)
            x_after_utm, y_after_utm = wse_after["x"].values, wse_after["y"].values
            X_after_utm, Y_after_utm = np.meshgrid(x_after_utm, y_after_utm)
            lon_after, lat_after = transformer_after_to_latlon.transform(X_after_utm, Y_after_utm)
            wse_before_ll = xr.DataArray(wse_before.values, coords={'lat': lat_before[:,0], 'lon': lon_before[0,:]}, dims=['y', 'x']).rename({'y': 'lat', 'x': 'lon'})
            wse_after_ll = xr.DataArray(wse_after.values, coords={'lat': lat_after[:,0], 'lon': lon_after[0,:]}, dims=['y', 'x']).rename({'y': 'lat', 'x': 'lon'})

            try:
                wse_before_interpolated = wse_before_ll.interp(lat=wse_after_ll.lat, lon=wse_after_ll.lon, method="linear")
                wse_diff = wse_after_ll - wse_before_interpolated
            except Exception as e:
                print(f"Error during interpolation for difference: {e}.")
                ax.set_title(f"{title}\n(Error in re-gridding)", fontsize=14)
                ax.set_visible(False)
                return None
            X_lon, Y_lat = lon_after, lat_after
        else:
            try:
                wse_before_aligned = wse_before.interp(x=wse_after.x, y=wse_after.y, method="linear")
                wse_diff = wse_after - wse_before_aligned
            except Exception as e:
                print(f"Error interpolating WSE 'before' onto 'after' grid: {e}. Assuming perfect alignment.")
                if wse_before.shape == wse_after.shape and np.allclose(wse_before['x'], wse_after['x']) and np.allclose(wse_before['y'], wse_after['y']):
                    wse_diff = wse_after - wse_before
                else:
                    print("Direct difference not possible due to incompatible grids. Skipping difference plot.")
                    ax.set_title(f"{title}\n(Incompatible data grids)", fontsize=14)
                    ax.set_visible(False)
                    return None
            utm_crs_plot = ccrs.UTM(zone=ds_before.utm_zone_num, southern_hemisphere=False)
            transformer = Transformer.from_crs(utm_crs_plot, ccrs.PlateCarree(), always_xy=True)
            x_utm, y_utm = wse_after["x"].values, wse_after["y"].values
            X_utm, Y_utm = np.meshgrid(x_utm, y_utm)
            X_lon, Y_lat = transformer.transform(X_utm, Y_utm)

        mesh = ax.pcolormesh(
            X_lon,
            Y_lat,
            wse_diff.values,
            transform=ccrs.PlateCarree(),
            cmap="RdBu",
            vmin=vmin,
            vmax=vmax,
        )
        ax.gridlines(draw_labels=True)
        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
        ax.coastlines(resolution='10m', linewidth=1)
        ax.set_title(title, fontsize=14)
        return mesh

    def run_analysis(self):
        """Main method to run the analysis, prompting the user for input."""
        num_pairs = int(input("How many SWOT granule pairs (before and after) do you want to analyze? "))

        for i in range(num_pairs):
            print(f"\n--- Collecting data for Pair {i+1} ---")
            ds_before = self._get_granule_data("BEFORE")
            ds_after = self._get_granule_data("AFTER")
            
            if ds_before is not None and ds_after is not None:
                self.swot_datasets.append((ds_before, ds_after))
            elif ds_before is None and ds_after is None:
                print(f"Skipping Pair {i+1} as no granules were found for both before and after.")
            elif ds_before is None:
                print(f"Skipping Pair {i+1} as no 'before' granule was found.")
            else:
                print(f"Skipping Pair {i+1} as no 'after' granule was found.")
        
        if not self.swot_datasets:
            print("No valid granule pairs were found or entered. Exiting.")
            return

        num_rows = len(self.swot_datasets)
        num_cols = 3
        fig = plt.figure(figsize=(24, 8 * num_rows))
        gs = fig.add_gridspec(num_rows, num_cols + 2, width_ratios=[1, 1, 1, 0.05, 0.05], wspace=0.5)
        main_location = input("Enter a main location title for all plots: ")

        for i, (ds_before, ds_after) in enumerate(self.swot_datasets):
            print("\n--- Diagnostic Check ---")
            print(f"Dataset for 'BEFORE' event (Pair {i+1}): {type(ds_before)}")
            print(f"Dataset for 'AFTER' event (Pair {i+1}): {type(ds_after)}")
            print("------------------------")

        for i, (ds_before, ds_after) in enumerate(self.swot_datasets):
            print(f"\n--- Plotting for Pair {i+1} ---")
            title_before = input(f"Title for Before Event (Pair {i+1}): ")
            title_after = input(f"Title for After Event (Pair {i+1}): ")
            title_diff = input(f"Title for Difference Plot (Pair {i+1}): ")

            ax1 = fig.add_subplot(gs[i, 0], projection=ccrs.PlateCarree())
            mesh1, wse1 = self._plot_swot_data(ax1, ds_before, title_before, vmin=0, vmax=10)
            ax2 = fig.add_subplot(gs[i, 1], projection=ccrs.PlateCarree())
            mesh2, wse2 = self._plot_swot_data(ax2, ds_after, title_after, vmin=0, vmax=10)
            ax3 = fig.add_subplot(gs[i, 2], projection=ccrs.PlateCarree())
            mesh3 = self._plot_difference(ax3, ds_before, ds_after, title_diff, vmin=-5, vmax=5)

            if mesh1:
                cax_wse = fig.add_subplot(gs[i, num_cols])
                cbar_wse = fig.colorbar(mesh1, cax=cax_wse, orientation='vertical')
                cbar_wse.set_label("Water Surface Elevation [m]", fontsize=12, rotation=90)
            if mesh3:
                cax_diff = fig.add_subplot(gs[i, num_cols + 1])
                cbar_diff = fig.colorbar(mesh3, cax=cax_diff, orientation='vertical')
                cbar_diff.set_label("WSE Difference [m]", fontsize=12, rotation=90)

        fig.text(x=0.5, y=0.98, s=main_location, fontsize=20, color='black', va='top', ha='center')
        plt.savefig('West coast of India (Ponnani).pdf')
        plt.show()

# --- MAIN EXECUTION BLOCK ---
if __name__ == "__main__":
    analyzer = SWOTAnalyzer()
    analyzer.run_analysis()

How many SWOT granule pairs (before and after) do you want to analyze?  1



--- Collecting data for Pair 1 ---

--- Enter BEFORE SWOT granule details ---


Granule name for BEFORE (e.g., SWOT_L2_HR_Raster_...):  SWOT_L2_HR_Raster_250m_UTM22J_N_x_x_x_013_533_051F_20240415T150120_20240415T150141_PIC0_01.nc
Start date for BEFORE (YYYY-MM-DD):  2024-04-15
End date for BEFORE (YYYY-MM-DD, same as start for single day):  2024-04-15


Number of granules found: 0
No granules found! Check your search parameters.

--- Enter AFTER SWOT granule details ---


Granule name for AFTER (e.g., SWOT_L2_HR_Raster_...):  SWOT_L2_HR_Raster_250m_UTM22J_N_x_x_x_014_533_051F_20240506T114623_20240506T114644_PIC0_01.nc
Start date for AFTER (YYYY-MM-DD):  2025-05-06
End date for AFTER (YYYY-MM-DD, same as start for single day):  2025-05-06


Number of granules found: 0
No granules found! Check your search parameters.
Skipping Pair 1 as no granules were found for both before and after.
No valid granule pairs were found or entered. Exiting.


<Figure size 640x480 with 0 Axes>